In [2]:
!pip install catboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor, Pool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv('/content/df_cafes_.csv')
df.head()

,name,type,cuisine,website,lat,lon,dgis_general_rating,dgis_general_review_count_with_stars,dgis_org_rating,dgis_org_review_count,...,nearest_gallery_name,other_attractions_100m,other_attractions_500m,other_attractions_1000m,parks_100m,parks_500m,parks_1000m,nearest_park,nearest_park_name,res_buildings_density_1000m
0,Salden's Taphouse,bar,NaN,1,55.765574,37.638976,4.9,129.0,4.8,252.0,...,Nadja Brykina Gallery,1,65,220,0,6,27,247.780090,Сквер «Огородная слобода»,132.416913
1,Com,cafe,vietnamese,0,55.768280,37.644618,3.5,12.0,4.5,164.0,...,Sare art,1,73,179,2,9,32,42.032631,(без названия),113.954939
2,Шоколадница,cafe,coffee_shop,1,55.738811,37.653006,2.4,170.0,3.1,10899.0,...,Краснохолмская,0,16,50,1,2,8,91.996504,(без названия),56.659160
3,Тануки,restaurant,japanese,1,55.728517,37.679155,4.9,527.0,4.8,24921.0,...,Здесь на Таганке,2,4,12,0,0,7,509.973891,(без названия),46.791553
4,Вареничная №1,restaurant,russian,1,55.732211,37.665123,4.1,201.0,4.1,3881.0,...,Краснохолмская,0,7,30,0,2,9,191.115200,(без названия),69.709865


In [ ]:
df.drop(['name', 'brand'], axis=1, inplace=True)
df.head()

,type,cuisine,website,lat,lon,dgis_general_rating,dgis_general_review_count_with_stars,dgis_org_rating,dgis_org_review_count,city,...,nearest_gallery_name,other_attractions_100m,other_attractions_500m,other_attractions_1000m,parks_100m,parks_500m,parks_1000m,nearest_park,nearest_park_name,res_buildings_density_1000m
0,bar,NaN,1,55.765574,37.638976,4.9,129.0,4.8,252.0,Москва,...,Nadja Brykina Gallery,1,65,220,0,6,27,247.780090,Сквер «Огородная слобода»,132.416913
1,cafe,vietnamese,0,55.768280,37.644618,3.5,12.0,4.5,164.0,Москва,...,Sare art,1,73,179,2,9,32,42.032631,(без названия),113.954939
2,cafe,coffee_shop,1,55.738811,37.653006,2.4,170.0,3.1,10899.0,Москва,...,Краснохолмская,0,16,50,1,2,8,91.996504,(без названия),56.659160
3,restaurant,japanese,1,55.728517,37.679155,4.9,527.0,4.8,24921.0,Москва,...,Здесь на Таганке,2,4,12,0,0,7,509.973891,(без названия),46.791553
4,restaurant,russian,1,55.732211,37.665123,4.1,201.0,4.1,3881.0,Москва,...,Краснохолмская,0,7,30,0,2,9,191.115200,(без названия),69.709865


In [ ]:
# Напишем функцию для создания таргета
def get_target(df, m = 20, alpha = 1, beta = 1):

  rating_col = "dgis_general_rating" # колонка с рейтингом кафе R_i
  reviews_col = "dgis_general_review_count_with_stars" # колонка с числом отзывов v_i

  # WR
  R = df[rating_col]
  v = df[reviews_col].fillna(0)

  C = R.mean() # средний рейтинг по всем кафе

  df["WR"] = (v / (v + m)) * R + (m / (v + m)) * C

  # Vol_i = log(1 + v_i)
  df["Vol"] = np.log1p(v) # log(1 + v_i)

  # cтандартизация WR и Vol
  WR_mean = df["WR"].mean()
  WR_std  = df["WR"].std()

  Vol_mean = df["Vol"].mean()
  Vol_std  = df["Vol"].std()

  df["zWR"]  = (df["WR"]  - WR_mean)  / WR_std
  df["zVol"] = (df["Vol"] - Vol_mean) / Vol_std

  # итоговый таргет
  df["cafe_success_index"] = alpha * df["zWR"] + beta * df["zVol"]

  return df

In [ ]:
df_cafes = get_target(df)

# уберем лишние колонки
df_cafes.drop(columns = ["zWR", "WR", "zVol", "Vol", "dgis_general_rating", "dgis_org_rating",
                         "dgis_general_review_count_with_stars", "dgis_org_review_count", "city"], inplace = True)
df_cafes['cafe_success_index'].sample(5)

,cafe_success_index
772,3.920928
50,0.884820
3599,-0.453928
7075,-1.763584
5035,1.205438


In [ ]:
X = df_cafes.drop(columns=['cafe_success_index'])
y = df_cafes['cafe_success_index']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

**Baseline**

In [ ]:
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb = CatBoostRegressor(random_state=42, verbose=0)

In [ ]:
cb = CatBoostRegressor(random_state=42, verbose=0, early_stopping_rounds=50)
cb.fit(train_pool, eval_set=test_pool)

y_pred = cb.predict(test_pool)

In [ ]:
# для дальнейших сравнений результатов моделей создадим словарь
benchmark = {}

# оцениваем качество
y_pred_test = cb.predict(test_pool)
y_pred_train = cb.predict(train_pool)

# R^2
r2_test = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)

# MSE
mse_test = mean_squared_error(y_test, y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)

# RMSE = sqrt(MSE)
rmse_test = np.sqrt(mse_test)
rmse_train = np.sqrt(mse_train)

# MAE
mae_test = mean_absolute_error(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
# добавляем в бенчмарк для сравнения моделей
benchmark['catboost_baseline'] = [r2_test, mse_test, rmse_test, mae_test]

print("TEST:")
print("R^2 (test):", r2_test)
print("MSE (test):", mse_test)
print("RMSE(test):", rmse_test)
print("MAE (test):", mae_test)

print("\nTRAIN:")
print("R^2 (train):", r2_train)
print("MSE (train):", mse_train)
print("RMSE(train):", rmse_train)
print("MAE (train):", mae_train)

TEST:
R^2 (test): 0.23254236990763166
MSE (test): 2.024583106029105
RMSE(test): 1.4228784579257305
MAE (test): 1.1181894959880294

TRAIN:
R^2 (train): 0.30378378848160537
MSE (train): 1.8948552987493887
RMSE(train): 1.3765374309292824
MAE (train): 1.0910818359272025


**CatBoost + GridSearch**

In [ ]:
grid = {
    'depth': [3, 4, 5, 6],
    'l2_leaf_reg': [1, 3, 5, 10],
    'learning_rate': [0.01, 0.03, 0.1]
}

cb = CatBoostRegressor(iterations=1000, early_stopping_rounds=50,
                        random_state=42, verbose=0)

results = cb.grid_search(grid, train_pool, verbose=0)


bestTest = 1.466437298
bestIteration = 994

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.462989776
bestIteration = 483

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.457378836
bestIteration = 292


bestTest = 1.465947936
bestIteration = 993

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.460523597
bestIteration = 750

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.462314546
bestIteration = 198


bestTest = 1.465619156
bestIteration = 987


bestTest = 1.459611786
bestIteration = 977

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.462760684
bestIteration = 229


bestTest = 1.466632355
bestIteration = 995

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.463322252
bestIteration = 700

Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.462733776
bestIteration = 168


bestTest = 1.462180876
bestIteration = 999

Stopped by overfitting detector  (50 

In [ ]:
best_params = results['params']
print(best_params)

{'depth': 6, 'learning_rate': 0.1, 'l2_leaf_reg': 1}


In [ ]:
cb_best = CatBoostRegressor(
    **best_params,
    iterations=1000,
    early_stopping_rounds=50,
    random_state=42,
    verbose=0
)

cb_best.fit(train_pool, eval_set=test_pool)

CatBoostRegressor(depth=6, early_stopping_rounds=50, iterations=1000, l2_leaf_reg=1, learning_rate=0.1, loss_function='RMSE', random_state=42, verbose=0)

In [ ]:
# оцениваем качество
y_pred_test = cb_best.predict(test_pool)
y_pred_train = cb_best.predict(train_pool)

# R^2
r2_test = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)

# MSE
mse_test = mean_squared_error(y_test, y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)

# RMSE = sqrt(MSE)
rmse_test = np.sqrt(mse_test)
rmse_train = np.sqrt(mse_train)

# MAE
mae_test = mean_absolute_error(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
# добавляем в бенчмарк для сравнения моделей
benchmark['catboost_after_gridsearch'] = [r2_test, mse_test, rmse_test, mae_test]

print("TEST:")
print("R^2 (test):", r2_test)
print("MSE (test):", mse_test)
print("RMSE(test):", rmse_test)
print("MAE (test):", mae_test)

print("\nTRAIN:")
print("R^2 (train):", r2_train)
print("MSE (train):", mse_train)
print("RMSE(train):", rmse_train)
print("MAE (train):", mae_train)

TEST:
R^2 (test): 0.23343446180990857
MSE (test): 2.022229732860924
RMSE(test): 1.4220512412922832
MAE (test): 1.1195387004967974

TRAIN:
R^2 (train): 0.3039197204950155
MSE (train): 1.8944853396883694
RMSE(train): 1.3764030440566344
MAE (train): 1.0896254790170394


**Feature engineering**

In [ ]:
drop_cols = ['name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name']
# оставляем только те, что реально есть в данных
drop_cols = [c for c in drop_cols if c in df_cafes.columns]
df_cafes.drop(columns=drop_cols, inplace=True)

In [ ]:
df_cafes.head()

,type,cuisine,lat,lon,direct_competitors_500m,direct_competitors_1000m,direct_competitors_100m,indirect_competitors_500m,indirect_competitors_1000m,indirect_competitors_100m,...,nearest_gallery,other_attractions_100m,other_attractions_500m,other_attractions_1000m,parks_100m,parks_500m,parks_1000m,nearest_park,res_buildings_density_1000m,cafe_success_index
0,bar,NaN,55.765574,37.638976,0,0,0,95,408,13,...,288.163295,1,65,220,0,6,27,247.780090,132.416913,1.998754
1,cafe,vietnamese,55.768280,37.644618,1,3,0,47,293,5,...,346.831736,1,73,179,2,9,32,42.032631,113.954939,-1.612387
2,cafe,coffee_shop,55.738811,37.653006,20,51,2,69,149,0,...,305.446600,0,16,50,1,2,8,91.996504,56.659160,-3.288294
3,restaurant,japanese,55.728517,37.679155,0,0,0,9,29,0,...,1459.137150,2,4,12,0,0,7,509.973891,46.791553,3.067306
4,restaurant,russian,55.732211,37.665123,0,0,0,30,85,7,...,984.161965,0,7,30,0,2,9,191.115200,69.709865,0.563696


In [ ]:
df_cafes.select_dtypes(include='object').columns.tolist()

['type', 'cuisine']

In [ ]:
X = df_cafes.drop(columns=['cafe_success_index'])
y = df_cafes['cafe_success_index']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

In [ ]:
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb = CatBoostRegressor(random_state=42, verbose=0)

In [ ]:
cb = CatBoostRegressor(random_state=42, verbose=0, early_stopping_rounds=50)
cb.fit(train_pool, eval_set=test_pool)


In [ ]:
# для дальнейших сравнений результатов моделей создадим словарь
benchmark = {}

# оцениваем качество
y_pred_test = cb.predict(test_pool)
y_pred_train = cb.predict(train_pool)

# R^2
r2_test = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)

# MSE
mse_test = mean_squared_error(y_test, y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)

# RMSE = sqrt(MSE)
rmse_test = np.sqrt(mse_test)
rmse_train = np.sqrt(mse_train)

# MAE
mae_test = mean_absolute_error(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
# добавляем в бенчмарк для сравнения моделей
benchmark['catboost_baseline'] = [r2_test, mse_test, rmse_test, mae_test]

print("TEST:")
print("R^2 (test):", r2_test)
print("MSE (test):", mse_test)
print("RMSE(test):", rmse_test)
print("MAE (test):", mae_test)

print("\nTRAIN:")
print("R^2 (train):", r2_train)
print("MSE (train):", mse_train)
print("RMSE(train):", rmse_train)
print("MAE (train):", mae_train)

TEST:
R^2 (test): 0.2198295157256548
MSE (test): 2.058120110284498
RMSE(test): 1.4346149693504868
MAE (test): 1.1327827881279178

TRAIN:
R^2 (train): 0.28815383469352174
MSE (train): 1.9373945276046993
RMSE(train): 1.3919032033890502
MAE (train): 1.1049258152168913


In [ ]:
# общее количество инфраструктуры вокруг (проходимость)
df_cafes['total_infrastructure_500m'] = (
    df_cafes['bus_trolley_stops_500m'] +
    df_cafes['metro_stations_500m'] +
    df_cafes['malls_500m'] +
    df_cafes['business_centres_500m'] +
    df_cafes['universities & colleges_500m']
)

# отношение прямых конкурентов к общей инфраструктуре
df_cafes['competition_density_500m'] = (
    df_cafes['direct_competitors_500m'] /
    (df_cafes['total_infrastructure_500m'] + 1)
)

# доля своего бренда среди конкурентов
df_cafes['brand_share_500m'] = (
    df_cafes['same_brand_500m'] /
    (df_cafes['direct_competitors_500m'] + 1)
)

# культурный кластер
df_cafes['culture_500m'] = (
    df_cafes['museums_500m'] +
    df_cafes['theatres_500m'] +
    df_cafes['galleries_500m'] +
    df_cafes['other_attractions_500m']
)

In [ ]:
X = df_cafes.drop(columns=['cafe_success_index'])
y = df_cafes['cafe_success_index']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

In [ ]:
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb = CatBoostRegressor(random_state=42, verbose=0, early_stopping_rounds=50)
cb.fit(train_pool, eval_set=test_pool)

# оцениваем качество
y_pred_test = cb.predict(test_pool)
y_pred_train = cb.predict(train_pool)

# R^2
r2_test = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)

# MSE
mse_test = mean_squared_error(y_test, y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)

# RMSE = sqrt(MSE)
rmse_test = np.sqrt(mse_test)
rmse_train = np.sqrt(mse_train)

# MAE
mae_test = mean_absolute_error(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
# добавляем в бенчмарк для сравнения моделей
benchmark['catboost_baseline'] = [r2_test, mse_test, rmse_test, mae_test]

print("TEST:")
print("R^2 (test):", r2_test)
print("MSE (test):", mse_test)
print("RMSE(test):", rmse_test)
print("MAE (test):", mae_test)

print("\nTRAIN:")
print("R^2 (train):", r2_train)
print("MSE (train):", mse_train)
print("RMSE(train):", rmse_train)
print("MAE (train):", mae_train)

TEST:
R^2 (test): 0.21897830773470417
MSE (test): 2.060365630102951
RMSE(test): 1.4353973770712245
MAE (test): 1.1309101084763329

TRAIN:
R^2 (train): 0.2954789828441037
MSE (train): 1.9174580544838236
RMSE(train): 1.384723096681724
MAE (train): 1.1001039804135229


поиграем с коэффициентами таргета

In [ ]:
df = pd.read_csv('/content/df_cafes_.csv')
df.drop(['name', 'brand'], axis=1, inplace=True)
df_cafes = get_target(df, m=20, alpha=1, beta=2)
# уберем лишние колонки
df_cafes.drop(columns = [], inplace = True)
drop_cols = ["zWR", "WR", "zVol", "Vol", "dgis_general_rating", "dgis_org_rating",
             "dgis_general_review_count_with_stars", "dgis_org_review_count", "city",
             'name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name']

drop_cols = [c for c in drop_cols if c in df_cafes.columns]
df_cafes.drop(columns=drop_cols, inplace=True)


In [ ]:
X = df_cafes.drop(columns=['cafe_success_index'])
y = df_cafes['cafe_success_index']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

In [ ]:
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

grid = {
    'depth': [3, 4, 5, 6],
    'l2_leaf_reg': [1, 3, 5, 10],
    'learning_rate': [0.01, 0.03, 0.1]
}

cb = CatBoostRegressor(iterations=1000, early_stopping_rounds=50,
                        random_state=42, verbose=0)

results = cb.grid_search(grid, train_pool, verbose=0)


bestTest = 2.225842803
bestIteration = 985

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.212650099
bestIteration = 677

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.216775149
bestIteration = 249


bestTest = 2.223783208
bestIteration = 998

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.215444156
bestIteration = 808

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.211699189
bestIteration = 324


bestTest = 2.220992641
bestIteration = 999

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.220239688
bestIteration = 509

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.217192865
bestIteration = 244


bestTest = 2.223556425
bestIteration = 999

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.216569862
bestIteration = 599

Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.212802234
bestIteration = 229


bestTest = 2.214158685
best

In [ ]:
best_params = results['params']
print(best_params)

{'depth': 5, 'learning_rate': 0.03, 'l2_leaf_reg': 1}


In [ ]:
cb_best = CatBoostRegressor(
    **best_params,
    iterations=1000,
    early_stopping_rounds=50,
    random_state=42,
    verbose=0
)

cb_best.fit(train_pool, eval_set=test_pool)

CatBoostRegressor(depth=5, early_stopping_rounds=50, iterations=1000, l2_leaf_reg=1, learning_rate=0.03, loss_function='RMSE', random_state=42, verbose=0)

In [ ]:
# оцениваем качество
y_pred_test = cb_best.predict(test_pool)
y_pred_train = cb_best.predict(train_pool)

# R^2
r2_test = r2_score(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)

# MSE
mse_test = mean_squared_error(y_test, y_pred_test)
mse_train = mean_squared_error(y_train, y_pred_train)

# RMSE = sqrt(MSE)
rmse_test = np.sqrt(mse_test)
rmse_train = np.sqrt(mse_train)

# MAE
mae_test = mean_absolute_error(y_test, y_pred_test)
mae_train = mean_absolute_error(y_train, y_pred_train)
# добавляем в бенчмарк для сравнения моделей
benchmark['catboost_after_gridsearch'] = [r2_test, mse_test, rmse_test, mae_test]

print("TEST:")
print("R^2 (test):", r2_test)
print("MSE (test):", mse_test)
print("RMSE(test):", rmse_test)
print("MAE (test):", mae_test)

print("\nTRAIN:")
print("R^2 (train):", r2_train)
print("MSE (train):", mse_train)
print("RMSE(train):", rmse_train)
print("MAE (train):", mae_train)

TEST:
R^2 (test): 0.25370850415366064
MSE (test): 4.653592284776915
RMSE(test): 2.1572186455658393
MAE (test): 1.6978579274989611

TRAIN:
R^2 (train): 0.33568797686276963
MSE (train): 4.286998067861853
RMSE(train): 2.0705067176567797
MAE (train): 1.6417965686208855


In [ ]:
df = pd.read_csv('/content/df_cafes_.csv')
y_vol = np.log1p(df['dgis_general_review_count_with_stars'].fillna(0))

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X, y_vol, test_size=0.2, random_state=42)
X_train_v[cat_cols] = X_train_v[cat_cols].fillna('missing')
X_test_v[cat_cols] = X_test_v[cat_cols].fillna('missing')
train_pool_v = Pool(X_train_v, y_train_v, cat_features=cat_cols)
test_pool_v = Pool(X_test_v, y_test_v, cat_features=cat_cols)

cb_vol = CatBoostRegressor(
    **best_params,
    iterations=1000,
    early_stopping_rounds=50,
    random_state=42,
    verbose=0
)

cb_vol.fit(train_pool_v, eval_set=test_pool_v)

y_pred_v = cb_vol.predict(test_pool_v)
print('R^2 (Vol only):', r2_score(y_test_v, y_pred_v))

R^2 (Vol only): 0.2901834563142611


Попробуем брать теперь только y_vol, чисто ради эксперимента

In [ ]:
df = pd.read_csv('/content/df_cafes_.csv')
y_vol = np.log1p(df['dgis_general_review_count_with_stars'].fillna(0))
df.drop(['name', 'brand'], axis=1, inplace=True)
df_cafes = get_target(df)

# уберем лишние колонки
drop_cols = ["zWR", "WR", "zVol", "Vol", "dgis_general_rating", "dgis_org_rating",
             "dgis_general_review_count_with_stars", "dgis_org_review_count", "city",
             'name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name']

drop_cols = [c for c in drop_cols if c in df_cafes.columns]
df_cafes.drop(columns=drop_cols, inplace=True)



# ПРОХОДИМОСТЬ

# суммарная транспортная доступность
df_cafes['transport_access_500m'] = (
    df_cafes['bus_trolley_stops_500m'] +
    df_cafes['metro_stations_500m'] * 5  # метро весит больше чем автобус
)

# близость к метро (обратная величина - чем ближе, тем больше)
df_cafes['metro_proximity'] = 1 / (df_cafes['nearest_metro_station'] + 1)

# ПЛОТНОСТЬ ЛЮДЕЙ ВОКРУГ

# потенциальная аудитория: жильё + работа + учёба
df_cafes['audience_500m'] = (
    df_cafes['res_buildings_density_1000m'] +
    df_cafes['business_centres_500m'] * 3 +
    df_cafes['universities & colleges_500m'] * 5
)

# КОНКУРЕНТНАЯ СРЕДА

# плотность всех заведений общепита (прямые + косвенные)
df_cafes['total_food_500m'] = (
    df_cafes['direct_competitors_500m'] +
    df_cafes['indirect_competitors_500m']
)

# доля прямых конкурентов среди всех
df_cafes['direct_share_500m'] = (
    df_cafes['direct_competitors_500m'] /
    (df_cafes['total_food_500m'] + 1)
)

# конкуренция на единицу трафика
df_cafes['competitors_per_transport'] = (
    df_cafes['total_food_500m'] /
    (df_cafes['transport_access_500m'] + 1)
)

# ПРИВЛЕКАТЕЛЬНОСТЬ РАЙОНА

# туристический потенциал
df_cafes['tourist_score_500m'] = (
    df_cafes['museums_500m'] * 3 +
    df_cafes['theatres_500m'] * 2 +
    df_cafes['galleries_500m'] +
    df_cafes['other_attractions_500m'] * 2 +
    df_cafes['parks_500m']
)

# центральность (обратная дистанция до центра)
df_cafes['centrality'] = 1 / (df_cafes['distance_to_city_centre'] + 1)

#СЕТЕВОЙ ЭФФЕКТ

# сетевое заведение или нет (brand уже дропнут, но same_brand остался)
df_cafes['is_chain'] = (df_cafes['same_brand_1000m'] > 0).astype(int)

# размер сети вокруг
df_cafes['chain_presence'] = (
    df_cafes['same_brand_500m'] +
    df_cafes['same_brand_1000m']
)

#МОНОПОЛИЯ НА РЫНКЕ

# обратный HHI - чем ниже концентрация, тем разнообразнее рынок
df_cafes['market_diversity_500m'] = 1 / (df_cafes['hhindex_500m'] + 0.01)

#КОМБИНАЦИИ РАДИУСОВ

# рост конкуренции от 100м к 1000м (насколько "загруженный" район)
df_cafes['competition_gradient'] = (
    df_cafes['direct_competitors_1000m'] -
    df_cafes['direct_competitors_100m']
)

#ЛОГАРИФМЫ ДИСТАНЦИЙ

dist_cols = ['nearest_metro_station', 'nearest_bus_trolley_stop',
             'nearest_railway_station', 'nearest_business_centre',
             'nearest_museum', 'nearest_theatre', 'nearest_gallery',
             'nearest_park', 'distance_to_city_centre']
dist_cols = [c for c in dist_cols if c in df_cafes.columns]
for col in dist_cols:
    df_cafes[f'{col}_log'] = np.log1p(df_cafes[col])

In [ ]:
df_cafes.columns

Index(['type', 'cuisine', 'lat', 'lon', 'direct_competitors_500m',
       'direct_competitors_1000m', 'direct_competitors_100m',
       'indirect_competitors_500m', 'indirect_competitors_1000m',
       'indirect_competitors_100m', 'same_brand_100m', 'same_brand_500m',
       'same_brand_1000m', 'hhindex_100m', 'hhindex_500m', 'hhindex_1000m',
       'malls_100m', 'malls_500m', 'malls_1000m',
       'universities & colleges_100m', 'universities & colleges_500m',
       'universities & colleges_1000m', 'bus_trolley_stops_100m',
       'bus_trolley_stops_500m', 'bus_trolley_stops_1000m',
       'nearest_bus_trolley_stop', 'nearest_metro_station',
       'metro_stations_100m', 'metro_stations_500m', 'metro_stations_1000m',
       'railway_stations_100m', 'railway_stations_500m',
       'railway_stations_1000m', 'nearest_railway_station',
       'distance_to_city_centre', 'medical_facilities_100m',
       'medical_facilities_500m', 'medical_facilities_1000m',
       'schools_kindergartens_1

In [ ]:
X = df_cafes.drop(columns=['cafe_success_index'])

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X, y_vol, test_size=0.2, random_state=42)
X_train_v[cat_cols] = X_train_v[cat_cols].fillna('missing')
X_test_v[cat_cols] = X_test_v[cat_cols].fillna('missing')
train_pool_v = Pool(X_train_v, y_train_v, cat_features=cat_cols)
test_pool_v = Pool(X_test_v, y_test_v, cat_features=cat_cols)

In [ ]:
cb_vol = CatBoostRegressor(
    **best_params,
    iterations=1000,
    early_stopping_rounds=50,
    random_state=42,
    verbose=0
)

cb_vol.fit(train_pool_v, eval_set=test_pool_v)

y_pred_v = cb_vol.predict(test_pool_v)
y_pred_v_train = cb_vol.predict(train_pool_v)
print('Test R^2 (Vol only):', r2_score(y_test_v, y_pred_v))
print('Train R^2 (Vol only):', r2_score(y_train_v, y_pred_v_train))

Test R^2 (Vol only): 0.2841080125953549
Train R^2 (Vol only): 0.4262674303678553


In [ ]:
!pip install transliterate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.8 MB/s eta 0:00:00


In [ ]:
import requests
import json
import time
import pandas as pd
from tqdm import tqdm

session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Referer": "https://www.afisha.ru/msk/restaurants/restaurant_list/",
}

all_results = []
page = 1
total_pages = None

while True:
    resp = session.get(
        "https://www.afisha.ru/rests/api/public/v1/restaurant/",
        params={"city": 2, "page": page, "page_size": 25},
        headers=headers,
        timeout=15
    )

    if resp.status_code != 200:
        print(f"Page {page}: status {resp.status_code}, retrying...")
        time.sleep(5)
        continue

    data = resp.json()

    if total_pages is None:
        total_pages = data['num_pages']
        print(f"Всего: {data['count']} заведений, {total_pages} страниц")

    for item in data['results']:
        tags = item.get('tags', {}) or {}
        extra = item.get('extra_info', {}) or {}
        avg_bill_tags = tags.get('average_bill', {}) or {}
        avg_bill_extra = extra.get('average_bill', {}) or {}

        all_results.append({
            'afisha_id': item.get('id'),
            'afisha_name': item.get('name'),
            'afisha_lat': item.get('latitude'),
            'afisha_lon': item.get('longitude'),
            'afisha_type': item.get('restaurant_type'),
            'average_bill': avg_bill_tags.get('name') or avg_bill_extra.get('name'),
            'average_bill_alt': avg_bill_tags.get('alt_name') or avg_bill_extra.get('alt_name'),
            'open_date': item.get('open_date'),
            'work_time': extra.get('work_time'),
            'delivery': extra.get('delivery'),
            'parking': extra.get('parking'),
            'rating_afisha': item.get('rating'),
            'twogis_id': item.get('twogis_id'),
        })

    if page % 20 == 0:
        print(f"Page {page}/{total_pages}, собрано {len(all_results)}")

    if data.get('next') is None:
        break

    page += 1
    time.sleep(random.uniform(0.5, 1.0))

import random  # добавь в начало если нет

afisha_df = pd.DataFrame(all_results)
afisha_df.to_csv('/content/afisha_all.csv', index=False)

print(f"\nГотово! Скачано {len(afisha_df)} заведений")
print(f"С average_bill: {afisha_df['average_bill'].notna().sum()}")
print(f"С open_date: {afisha_df['open_date'].notna().sum()}")
print(f"\nРаспределение average_bill:")
print(afisha_df['average_bill'].value_counts())

Всего: 6854 заведений, 275 страниц
Page 20/275, собрано 500
Page 40/275, собрано 1000
Page 60/275, собрано 1500
Page 80/275, собрано 2000
Page 100/275, собрано 2500
Page 120/275, собрано 3000
Page 140/275, собрано 3500
Page 160/275, собрано 4000
Page 180/275, собрано 4500
Page 200/275, собрано 5000
Page 220/275, собрано 5500
Page 240/275, собрано 6000
Page 260/275, собрано 6500

Готово! Скачано 6854 заведений
С average_bill: 6854
С open_date: 1096

Распределение average_bill:
average_bill
1000–3000 рублей      3240
До 1000 рублей        1448
                      1292
3000–6000 рублей       763
Больше 6000 рублей     111
Name: count, dtype: int64


In [ ]:
afisha_df.to_csv('/content/afisha_all.csv', index=False, encoding='utf-8-sig')

**Обогащенные данные**

In [ ]:
from sklearn.neighbors import BallTree
import numpy as np

df = pd.read_csv('/content/df_cafes_.csv')

# координаты в радианах для BallTree
afisha_coords = np.radians(afisha_df[['afisha_lat', 'afisha_lon']].values)
df_coords = np.radians(df[['lat', 'lon']].values)

tree = BallTree(afisha_coords, metric='haversine')

# ищем ближайшее заведение, радиус Земли = 6371 км
distances, indices = tree.query(df_coords, k=1)
distances_meters = distances.flatten() * 6371000  # в метры

# берём данные ближайшего
matched = afisha_df.iloc[indices.flatten()].reset_index(drop=True)

df['afisha_average_bill'] = matched['average_bill'].values
df['afisha_open_date'] = matched['open_date'].values
df['afisha_work_time'] = matched['work_time'].values
df['afisha_delivery'] = matched['delivery'].values
df['afisha_parking'] = matched['parking'].values
df['afisha_match_distance'] = distances_meters

# проверяем качество матчинга
print(f"Средняя дистанция: {distances_meters.mean():.0f} м")
print(f"Медианная дистанция: {np.median(distances_meters):.0f} м")
print(f"Матчей < 100м: {(distances_meters < 100).sum()} из {len(df)}")
print(f"Матчей < 500м: {(distances_meters < 500).sum()} из {len(df)}")
print(f"\nЗаполненность average_bill: {df['afisha_average_bill'].notna().sum()}")
print(f"\nПримеры:")
print(df[['name', 'afisha_average_bill', 'afisha_match_distance']].head(20).to_string())

Средняя дистанция: 130 м
Медианная дистанция: 30 м
Матчей < 100м: 7040 из 9218
Матчей < 500м: 8762 из 9218

Заполненность average_bill: 9218

Примеры:
                 name afisha_average_bill  afisha_match_distance
0   Salden's Taphouse      До 1000 рублей              36.120533
1                 Com    1000–3000 рублей               9.548642
2         Шоколадница    1000–3000 рублей             109.348968
3              Тануки    1000–3000 рублей              19.317825
4       Вареничная №1    1000–3000 рублей               4.132797
5              Грабли      До 1000 рублей               8.743569
6    Вкусно — и точка    3000–6000 рублей              52.595583
7         Шоколадница    1000–3000 рублей              29.969609
8    Вкусно — и точка    1000–3000 рублей               9.648012
9            IL Патио      До 1000 рублей              34.421901
10           Rostic's                                  73.584675
11            Fridays      До 1000 рублей              39.475718
12  

In [ ]:
# отсекаем плохие матчи (> 100м для сетевых, > 200м для остальных)
bad_match = df['afisha_match_distance'] > 200
df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
                     'afisha_work_time', 'afisha_delivery', 'afisha_parking']] = None

print(f"Отфильтровано плохих матчей: {bad_match.sum()}")
print(f"Осталось с average_bill: {df['afisha_average_bill'].notna().sum()}")

# преобразуем average_bill в числовой признак (price_level)
bill_to_level = {
    'До 1000 рублей': 1,
    '1000–3000 рублей': 2,
    '3000–6000 рублей': 3,
    'От 6000 рублей': 4
}
df['price_level'] = df['afisha_average_bill'].map(bill_to_level)

# возраст заведения в днях
df['afisha_open_date'] = pd.to_datetime(df['afisha_open_date'], errors='coerce')
df['age_days'] = (pd.Timestamp.now() - df['afisha_open_date']).dt.days

print(f"\nРаспределение price_level:")
print(df['price_level'].value_counts().sort_index())
print(f"\nЗаполненность age_days: {df['age_days'].notna().sum()}")
print(f"Медианный возраст: {df['age_days'].median():.0f} дней")

Отфильтровано плохих матчей: 1243
Осталось с average_bill: 7975

Распределение price_level:
price_level
1.0    1773
2.0    3813
3.0     756
Name: count, dtype: int64

Заполненность age_days: 1291
Медианный возраст: 3982 дней


/tmp/ipykernel_6020/3668447926.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
/tmp/ipykernel_6020/3668447926.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',


In [ ]:
# подготовка данных
df_cafes = get_target(df, m=20, alpha=1, beta=2)

drop_cols = ["zWR", "WR", "zVol", "Vol", "dgis_general_rating", "dgis_org_rating",
             "dgis_general_review_count_with_stars", "dgis_org_review_count", "city",
             'name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name', 'brand',
             # убираем строковые колонки afisha
             'afisha_average_bill', 'afisha_open_date', 'afisha_work_time',
             'afisha_match_distance']

drop_cols = [c for c in drop_cols if c in df_cafes.columns]
df_cafes.drop(columns=drop_cols, inplace=True)



In [ ]:
# delivery и parking привести к числам
if 'afisha_delivery' in df_cafes.columns:
    df_cafes['afisha_delivery'] = df_cafes['afisha_delivery'].astype(float)
if 'afisha_parking' in df_cafes.columns:
    df_cafes['afisha_parking'] = df_cafes['afisha_parking'].astype(float)

# работаем с пропусками
df_cafes['price_level'] = df_cafes.groupby('type')['price_level'].transform(
    lambda x: x.fillna(x.median())
)
# оставшиеся NaN заполняем общей медианой
df_cafes['price_level'] = df_cafes['price_level'].fillna(df_cafes['price_level'].median())

X = df_cafes.drop(columns=['cafe_success_index'])
y = df_cafes['cafe_success_index']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb = CatBoostRegressor(depth = 5, learning_rate =0.03, l2_leaf_reg = 1, iterations=1000, early_stopping_rounds=50, random_state=42, verbose=100)
cb.fit(train_pool, eval_set=test_pool)

y_pred_test = cb.predict(test_pool)
y_pred_train = cb.predict(train_pool)

print("\nTEST:")
print("R^2:", r2_score(y_test, y_pred_test))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
print("\nTRAIN:")
print("R^2:", r2_score(y_train, y_pred_train))
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))

# важность новых признаков
importances = cb.get_feature_importance()
fi = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)
print("\nТоп-15 признаков:")
print(fi.head(15).to_string(index=False))

0:	learn: 2.5297370	test: 2.4853927	best: 2.4853927 (0)	total: 12.5ms	remaining: 12.5s
100:	learn: 2.2536712	test: 2.1838767	best: 2.1838767 (100)	total: 1.08s	remaining: 9.63s
200:	learn: 2.2009580	test: 2.1539658	best: 2.1537921 (197)	total: 2.08s	remaining: 8.29s
300:	learn: 2.1607585	test: 2.1464207	best: 2.1461479 (297)	total: 3.65s	remaining: 8.49s
400:	learn: 2.1206715	test: 2.1433557	best: 2.1432874 (398)	total: 5.76s	remaining: 8.6s
500:	learn: 2.0844828	test: 2.1417687	best: 2.1416183 (462)	total: 6.83s	remaining: 6.8s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 2.141115943
bestIteration = 521

Shrink model to first 522 iterations.

TEST:
R^2: 0.26480840381993465
RMSE: 2.141115946667758

TRAIN:
R^2: 0.35248176300486334
RMSE: 2.044168020835419

Топ-15 признаков:
                   feature  importance
                      type   15.175552
                   cuisine   14.260725
          same_brand_1000m    4.265353
           same_brand_500m    3.811440
 

In [ ]:
print(fi[fi['feature'] == 'price_level'])
print(f"\nNaN в price_level: {X_train['price_level'].isna().sum()} из {len(X_train)}")

        feature  importance
67  price_level    1.023587

NaN в price_level: 0 из 7374


In [ ]:
from sklearn.neighbors import BallTree
import numpy as np

df = pd.read_csv('/content/df_cafes_.csv')


# считаем характеристики бренда
brand_stats = df.groupby('brand').agg(
    brand_size=('brand', 'size'),  # сколько точек у бренда
    brand_mean_rating=('dgis_general_rating', 'mean'),  # средний рейтинг сети
).reset_index()

df = df.merge(brand_stats, on='brand', how='left')

# бинарный признак: сеть или нет
df['is_chain'] = (df['brand'] != 'single').astype(int)

# логарифм размера сети
df['brand_size_log'] = np.log1p(df['brand_size'])

# координаты в радианах для BallTree
afisha_coords = np.radians(afisha_df[['afisha_lat', 'afisha_lon']].values)
df_coords = np.radians(df[['lat', 'lon']].values)

tree = BallTree(afisha_coords, metric='haversine')

distances, indices = tree.query(df_coords, k=1)
distances_meters = distances.flatten() * 6371000

matched = afisha_df.iloc[indices.flatten()].reset_index(drop=True)

df['afisha_average_bill'] = matched['average_bill'].values
df['afisha_open_date'] = matched['open_date'].values
df['afisha_work_time'] = matched['work_time'].values
df['afisha_delivery'] = matched['delivery'].values
df['afisha_parking'] = matched['parking'].values
df['afisha_match_distance'] = distances_meters

bad_match = df['afisha_match_distance'] > 200
df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
                     'afisha_work_time', 'afisha_delivery', 'afisha_parking']] = None

bill_to_level = {
    'До 1000 рублей': 1,
    '1000–3000 рублей': 2,
    '3000–6000 рублей': 3,
    'От 6000 рублей': 4
}
df['price_level'] = df['afisha_average_bill'].map(bill_to_level)

df['afisha_open_date'] = pd.to_datetime(df['afisha_open_date'], errors='coerce')
df['age_days'] = (pd.Timestamp.now() - df['afisha_open_date']).dt.days

# таргет - Vol
y_vol = np.log1p(df['dgis_general_review_count_with_stars'].fillna(0))

# дропаем ненужные колонки
drop_cols = ["dgis_general_rating", "dgis_org_rating",
             "dgis_general_review_count_with_stars", "dgis_org_review_count", "city",
             'name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name', 'brand',
             'afisha_average_bill', 'afisha_open_date', 'afisha_work_time',
             'afisha_match_distance']

drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)

if 'afisha_delivery' in df.columns:
    df['afisha_delivery'] = df['afisha_delivery'].astype(float)
if 'afisha_parking' in df.columns:
    df['afisha_parking'] = df['afisha_parking'].astype(float)

df['price_level'] = df.groupby('type')['price_level'].transform(
    lambda x: x.fillna(x.median())
)
df['price_level'] = df['price_level'].fillna(df['price_level'].median())

X = df
X_train, X_test, y_train, y_test = train_test_split(X, y_vol, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb = CatBoostRegressor(depth=4, learning_rate=0.03, l2_leaf_reg=10,
                            min_data_in_leaf=30, subsample=0.8,
                            colsample_bylevel=0.8, iterations=2000,
                            early_stopping_rounds=50, random_state=42, verbose=100)
cb.fit(train_pool, eval_set=test_pool)

y_pred_test = cb.predict(test_pool)
y_pred_train = cb.predict(train_pool)

print("\nTEST:")
print("R^2:", r2_score(y_test, y_pred_test))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
print("\nTRAIN:")
print("R^2:", r2_score(y_train, y_pred_train))
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))

importances = cb.get_feature_importance()
fi = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)
print("\nТоп-15 признаков:")
print(fi.head(15).to_string(index=False))

/tmp/ipykernel_6020/2501281206.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
/tmp/ipykernel_6020/2501281206.py:40: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',


0:	learn: 1.5488174	test: 1.5322050	best: 1.5322050 (0)	total: 10.3ms	remaining: 20.6s
100:	learn: 1.3227352	test: 1.3099237	best: 1.3099237 (100)	total: 772ms	remaining: 14.5s
200:	learn: 1.2796279	test: 1.2765116	best: 1.2765116 (200)	total: 1.5s	remaining: 13.4s
300:	learn: 1.2533341	test: 1.2623276	best: 1.2623276 (300)	total: 2.17s	remaining: 12.3s
400:	learn: 1.2310183	test: 1.2545282	best: 1.2545267 (399)	total: 2.89s	remaining: 11.5s
500:	learn: 1.2142976	test: 1.2488857	best: 1.2488857 (500)	total: 3.62s	remaining: 10.8s
600:	learn: 1.2018271	test: 1.2465717	best: 1.2463808 (593)	total: 4.31s	remaining: 10s
700:	learn: 1.1917562	test: 1.2452393	best: 1.2451857 (693)	total: 5.06s	remaining: 9.37s
800:	learn: 1.1823505	test: 1.2442883	best: 1.2442512 (797)	total: 5.74s	remaining: 8.59s
900:	learn: 1.1726827	test: 1.2433811	best: 1.2432997 (892)	total: 6.44s	remaining: 7.85s
1000:	learn: 1.1639961	test: 1.2424555	best: 1.2424555 (1000)	total: 7.15s	remaining: 7.13s
1100:	learn: 1

In [ ]:
print("Распределение is_chain:")
print(df['is_chain'].value_counts())
print(f"\nМедианный brand_size для сетей: {df[df['is_chain']==1]['brand_size'].median()}")
print(f"\nТоп-10 сетей по размеру:")
print(brand_stats.sort_values('brand_size', ascending=False).head(10).to_string(index=False))

Распределение is_chain:
is_chain
1    5099
0    4119
Name: count, dtype: int64

Медианный brand_size для сетей: 20.0

Топ-10 сетей по размеру:
           brand  brand_size  brand_mean_rating  brand_total_reviews  brand_mean_reviews
          single        4119           4.290580            1041714.0          252.904588
     Бургер Кинг         199           4.265829              49196.0          247.216080
           Cofix         193           3.909845              18864.0           97.740933
Вкусно — и точка         190           3.361579              26616.0          140.084211
        Rostic's         181           3.712155              37674.0          208.143646
One Price Coffee         175           4.382286              11948.0           68.274286
     Шоколадница         154           3.287013              49049.0          318.500000
 Крошка Картошка         127           3.593701               4183.0           32.937008
      Додо Пицца         125           4.494400         

**Таргет log(1 + кол-во отзывов)**




In [7]:
# Таргет 1: log(1 + кол-во отзывов) - регрессия

df = pd.read_csv('/content/df_cafes_.csv')
afisha_df = pd.read_csv('/content/afisha_all.csv')

# таргет
y = np.log1p(df['dgis_general_review_count_with_stars'].fillna(0))

# brand-фичи
brand_stats = df.groupby('brand').agg(
    brand_size=('brand', 'size'),
    brand_mean_rating=('dgis_general_rating', 'mean')).reset_index()

df = df.merge(brand_stats, on='brand', how='left')
df['is_chain'] = (df['brand'] != 'single').astype(int)
df['brand_size_log'] = np.log1p(df['brand_size'])

# afisha-фичи (то что спарсил с афиши.ру)
afisha_coords = np.radians(afisha_df[['afisha_lat', 'afisha_lon']].values)
df_coords = np.radians(df[['lat', 'lon']].values)

tree = BallTree(afisha_coords, metric='haversine')
distances, indices = tree.query(df_coords, k=1)
distances_meters = distances.flatten() * 6371000

matched = afisha_df.iloc[indices.flatten()].reset_index(drop=True)
df['afisha_average_bill'] = matched['average_bill'].values
df['afisha_open_date'] = matched['open_date'].values
df['afisha_work_time'] = matched['work_time'].values
df['afisha_delivery'] = matched['delivery'].values
df['afisha_parking'] = matched['parking'].values
df['afisha_match_distance'] = distances_meters

# отсекаем плохие матчи
bad_match = df['afisha_match_distance'] > 200
df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
                    'afisha_work_time', 'afisha_delivery', 'afisha_parking']] = None

bill_to_level = {
    'До 1000 рублей': 1, '1000–3000 рублей': 2,
    '3000–6000 рублей': 3, 'От 6000 рублей': 4}
df['price_level'] = df['afisha_average_bill'].map(bill_to_level)

df['afisha_open_date'] = pd.to_datetime(df['afisha_open_date'], errors='coerce')
df['age_days'] = (pd.Timestamp.now() - df['afisha_open_date']).dt.days

# дропаем ненужное
drop_cols = ['dgis_general_rating', 'dgis_org_rating',
             'dgis_general_review_count_with_stars', 'dgis_org_review_count', 'city',
             'name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name', 'brand',
             'afisha_average_bill', 'afisha_open_date', 'afisha_work_time',
             'afisha_match_distance']
drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)

# типы и пропуски
for col in ['afisha_delivery', 'afisha_parking']:
    if col in df.columns:
        df[col] = df[col].astype(float)

df['price_level'] = df.groupby('type')['price_level'].transform(lambda x: x.fillna(x.median()))
df['price_level'] = df['price_level'].fillna(df['price_level'].median())

/tmp/ipykernel_1504/3841438564.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
/tmp/ipykernel_1504/3841438564.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',


In [8]:
# обучение
X = df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb_vol = CatBoostRegressor(
    depth=4, learning_rate=0.03, l2_leaf_reg=10,
    min_data_in_leaf=30, subsample=0.8, colsample_bylevel=0.8,
    iterations=2000, early_stopping_rounds=50,
    random_state=42, verbose=200)
cb_vol.fit(train_pool, eval_set=test_pool)

y_pred_test = cb_vol.predict(test_pool)
y_pred_train = cb_vol.predict(train_pool)

print("\nTEST:")
print("R^2:", r2_score(y_test, y_pred_test))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))
print("\nTRAIN:")
print("R^2:", r2_score(y_train, y_pred_train))
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_pred_train)))

importances = cb_vol.get_feature_importance()
fi = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)
print("\nTop-15 features:")
print(fi.head(15).to_string(index=False))

0:	learn: 1.5488174	test: 1.5322050	best: 1.5322050 (0)	total: 52.4ms	remaining: 1m 44s
200:	learn: 1.2796279	test: 1.2765116	best: 1.2765116 (200)	total: 997ms	remaining: 8.92s
400:	learn: 1.2310183	test: 1.2545282	best: 1.2545267 (399)	total: 1.89s	remaining: 7.53s
600:	learn: 1.2018271	test: 1.2465717	best: 1.2463808 (593)	total: 2.79s	remaining: 6.5s
800:	learn: 1.1823505	test: 1.2442883	best: 1.2442512 (797)	total: 5.21s	remaining: 7.79s
1000:	learn: 1.1639961	test: 1.2424555	best: 1.2424555 (1000)	total: 6.1s	remaining: 6.09s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.240883962
bestIteration = 1127

Shrink model to first 1128 iterations.

TEST:
R^2: 0.3504509384339485
RMSE: 1.240883955772254

TRAIN:
R^2: 0.46171157663215256
RMSE: 1.141809101401163

Top-15 features:
                   feature  importance
         brand_mean_rating   18.048480
                   cuisine   13.756740
                      type    8.664032
                brand_size    5.02482

Качество модели - R2=0.35 на тесте, то есть модель объясняет 35% дисперсии логарифма числа отзывов. Это заметно лучше, чем составной таргет (там R2~0.23-0.26), что говорит о том, что объём отзывов предсказуем по инфраструктурным фичам, а вот рейтинг (вкусовое качество) - скорее нет, он зашумляет таргет.
По train/test (0.46 vs 0.35) - виден умеренное переобучение, но не критичное. Early stopping сработал на 1127 итерации из 2000, регуляризация делает своё дело

Feature importance:
Топ-3 фичи (brand_mean_rating 18%, cuisine 14%, type 9%). Это значит, что сетевые рестораны с высоким рейтингом бренда предсказуемо получают больше отзывов просто потому что они известны
Дальше идут brand_size/brand_size_log/is_chain - та же история: сетевое заведение = трафик.
Геолокационные фичи (metro, competitors, attractions) важны, но вносят по 1-3% каждая. Афиша (parking, delivery) тоже в топ-15, значит обогащение данных помогло.
Главный вывод: log(1 + reviews) - более чистый и предсказуемый таргет, чем composite index. Модель по сути ловит зависимость известность бренда + тип кухни + локация -> число отзывов

**Таргет (кол-во отзывов >= кол-во отзывов.quantile(0.8)).astype(int)**

In [13]:
# таргет 2: классификация (top-20% по отзывам)

from sklearn.metrics import (classification_report, roc_auc_score,
                             f1_score, confusion_matrix)
from catboost import CatBoostClassifier

df = pd.read_csv('/content/df_cafes_.csv')
afisha_df = pd.read_csv('/content/afisha_all.csv')

# таргет: 1 если кол-во отзывов >= 80-й перцентиль
reviews = df['dgis_general_review_count_with_stars'].fillna(0)
threshold = reviews.quantile(0.8)
y = (reviews >= threshold).astype(int)
print(f"Порог (quantile 0.8): {threshold}")
print(f"Распределение классов:\n{y.value_counts()}\n")

# ниже будет та же предобработка, что и выше была
# brand-фичи
brand_stats = df.groupby('brand').agg(
    brand_size=('brand', 'size'),
    brand_mean_rating=('dgis_general_rating', 'mean')).reset_index()

df = df.merge(brand_stats, on='brand', how='left')
df['is_chain'] = (df['brand'] != 'single').astype(int)
df['brand_size_log'] = np.log1p(df['brand_size'])

#afisha-фичи
afisha_coords = np.radians(afisha_df[['afisha_lat', 'afisha_lon']].values)
df_coords = np.radians(df[['lat', 'lon']].values)

tree = BallTree(afisha_coords, metric='haversine')
distances, indices = tree.query(df_coords, k=1)
distances_meters = distances.flatten() * 6371000

matched = afisha_df.iloc[indices.flatten()].reset_index(drop=True)
df['afisha_average_bill'] = matched['average_bill'].values
df['afisha_open_date'] = matched['open_date'].values
df['afisha_work_time'] = matched['work_time'].values
df['afisha_delivery'] = matched['delivery'].values
df['afisha_parking'] = matched['parking'].values
df['afisha_match_distance'] = distances_meters

bad_match = df['afisha_match_distance'] > 200
df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
                    'afisha_work_time', 'afisha_delivery', 'afisha_parking']] = None

bill_to_level = {
    'До 1000 рублей': 1, '1000–3000 рублей': 2,
    '3000–6000 рублей': 3, 'От 6000 рублей': 4}
df['price_level'] = df['afisha_average_bill'].map(bill_to_level)

df['afisha_open_date'] = pd.to_datetime(df['afisha_open_date'], errors='coerce')
df['age_days'] = (pd.Timestamp.now() - df['afisha_open_date']).dt.days

#дропаем ненужное
drop_cols = ['dgis_general_rating', 'dgis_org_rating',
             'dgis_general_review_count_with_stars', 'dgis_org_review_count', 'city',
             'name', 'website', 'nearest_museum_name',
             'nearest_theatre_name', 'nearest_gallery_name',
             'nearest_park_name', 'brand',
             'afisha_average_bill', 'afisha_open_date', 'afisha_work_time',
             'afisha_match_distance']
drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)

#типы и пропуски
for col in ['afisha_delivery', 'afisha_parking']:
    if col in df.columns:
        df[col] = df[col].astype(float)

df['price_level'] = df.groupby('type')['price_level'].transform(lambda x: x.fillna(x.median()))
df['price_level'] = df['price_level'].fillna(df['price_level'].median())



Порог (quantile 0.8): 156.0
Распределение классов:
dgis_general_review_count_with_stars
0    7373
1    1845
Name: count, dtype: int64



/tmp/ipykernel_1504/2467043719.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',
/tmp/ipykernel_1504/2467043719.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[bad_match, ['afisha_average_bill', 'afisha_open_date',


In [14]:
# обучение
X = df
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
X_train[cat_cols] = X_train[cat_cols].fillna('missing')
X_test[cat_cols] = X_test[cat_cols].fillna('missing')

train_pool = Pool(X_train, y_train, cat_features=cat_cols)
test_pool = Pool(X_test, y_test, cat_features=cat_cols)

cb_clf = CatBoostClassifier(
    depth=4, learning_rate=0.03, l2_leaf_reg=10,
    min_data_in_leaf=30, subsample=0.8, colsample_bylevel=0.8,
    iterations=2000, early_stopping_rounds=50,
    auto_class_weights='Balanced',
    eval_metric='F1',
    random_state=42, verbose=200)
cb_clf.fit(train_pool, eval_set=test_pool)

y_pred = cb_clf.predict(test_pool)
y_proba = cb_clf.predict_proba(test_pool)[:, 1]

print("\n" + "="*50)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nConfusion matrix:\n{confusion_matrix(y_test, y_pred)}")

importances = cb_clf.get_feature_importance()
fi = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)
print("\nTop-15 features:")
print(fi.head(15).to_string(index=False))

0:	learn: 0.6988356	test: 0.6915983	best: 0.6915983 (0)	total: 9.86ms	remaining: 19.7s
200:	learn: 0.7384423	test: 0.7151467	best: 0.7156321 (199)	total: 1.63s	remaining: 14.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.7156320741
bestIteration = 199

Shrink model to first 200 iterations.

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1471
           1       0.40      0.71      0.51       373

    accuracy                           0.73      1844
   macro avg       0.65      0.72      0.66      1844
weighted avg       0.80      0.73      0.75      1844

ROC-AUC: 0.7983

Confusion matrix:
[[1074  397]
 [ 110  263]]

Top-15 features:
                  feature  importance
                  cuisine   17.820357
        brand_mean_rating   16.076271
                     type    5.933282
               brand_size    5.123551
           brand_size_log    4.432291
          afisha_delivery 

ROC-AUC=0.80 - хороший результат для такой задачи
Для класса 1 (топ-20%) recall=0.71, т.е. модель находит 71% реально популярных заведений, но precision=0.40 -> из тех, кого она назвала популярными, только 40% таковыми являются. Это норм при Balanced весах - модель перестраховывается и предсказывает больше единиц (397 false positives)

**Feature importance **- картина почти идентична регрессии: cuisine (18%) и brand_mean_rating (16%) доминируют, дальше тип/размер бренда/сетевое заведение. Это подтверждает вывод: главные предикторы популярности - бренд, кухня, а не локация. Ну как, локация (metro, competitors, attractions) вносит по 2-3% каждая - важна, но вторична
Сравнение с регрессией: по сути модель ловит те же паттерны
Классификацию проще объяснять как "это заведение с 80% вероятностью попадёт в топ-20%"
Ранняя остановка на 199 итерации говорит о том, что задача проще для модели, чем регрессия - бинарное разделение схватывается быстро